In [ ]:
import boto3, botocore
from botocore.exceptions import ClientError
import os, time, json, io, zipfile, base64
from datetime import date
from dotenv import load_dotenv


from misc import load_from_yaml, save_to_yaml
import iam, s3, lf, rds, vpc, ec2

load_dotenv(".env")
# boto3.setup_default_session(profile_name="AMominNJ")

True

In [ ]:
ALL_IN_ONE_SG     = 'sg-0d8a868137f653df6'
ACCOUNT_ID        = os.environ['AWS_ACCOUNT_ID_ROOT']
REGION            = os.environ['AWS_DEFAULT_REGION']
VPC_ID            = os.environ['AWS_DEFAULT_VPC']
SECURITY_GROUP_ID = os.environ['AWS_DEFAULT_SG_ID']
SUBNET_IDS        = SUBNET_IDS = os.environ["AWS_DEFAULT_SUBNET_IDS"].split(":")
SUBNET_ID         = SUBNET_IDS[0]
AWS_INSTANCE_ID_JMASTER   = os.environ['AWS_INSTANCE_ID_JMASTER']
AWS_INSTANCE_ID_JAGENT    = os.environ['AWS_INSTANCE_ID_JAGENT']
AWS_DEFAULT_IMAGE_ID      = os.environ['AWS_DEFAULT_IMAGE_ID']
AWS_DEFAULT_KEY_PAIR_NAME = os.environ['AWS_DEFAULT_KEY_PAIR_NAME']
AWS_DEFAULT_INSTANCE_TYPE = os.environ['AWS_DEFAULT_INSTANCE_TYPE']

In [ ]:
ec2_client           = boto3.client('ec2', region_name=REGION)
ec2_resource         = boto3.resource('ec2', region_name=REGION)
rds_client = boto3.client("rds")
s3_client = boto3.client("s3")
iam_client = boto3.client("iam")
elbv2_client = boto3.client("elbv2", region_name=REGION)
acm_client = boto3.client("acm", region_name=REGION)  # Change region as needed

##### VPC

In [ ]:
vpc_cidr_block = '10.0.0.0/16'
vpc_name = 'ritual-roast-vpc'
vpc_id = ec2_client.create_vpc(CidrBlock=vpc_cidr_block)['Vpc']['VpcId']
# vpc = ec2_resource.Vpc('vpc_id')


##### Subnets

In [ ]:
subnet_configs = [
    {'cidr_block': '10.0.1.0/24', 'az': 'us-east-1a', 'tag': 'rr-public-subnet01-us-east-1a'},
    {'cidr_block': '10.0.2.0/24', 'az': 'us-east-1b', 'tag': 'rr-public-subnet02-us-east-1b'},
    {'cidr_block': '10.0.10.0/24', 'az': 'us-east-1a', 'tag': 'rr-app-subnet01-us-east-1a'},
    {'cidr_block': '10.0.11.0/24', 'az': 'us-east-1b', 'tag': 'rr-app-subnet02-us-east-1b'},
    {'cidr_block': '10.0.20.0/24', 'az': 'us-east-1a', 'tag': 'rr-data-subnet01-us-east-1a'},
    {'cidr_block': '10.0.21.0/24', 'az': 'us-east-1b', 'tag': 'rr-data-subnet02-us-east-1b'}
]

# Enable DNS Hostnames for the VPC
# Enable DNS Resolution for the VPC

In [ ]:
public_subnet1 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[0]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[0]['az']
)
ec2_client.create_tags(Resources=[public_subnet1.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[0]['tag']}])

public_subnet2 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[1]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[1]['az']
)
ec2_client.create_tags(Resources=[public_subnet2.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[1]['tag']}])

In [ ]:
app_subnet1 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[2]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[2]['az']
)
ec2_client.create_tags(Resources=[app_subnet1.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[2]['tag']}])

app_subnet2 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[3]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[3]['az']
)
ec2_client.create_tags(Resources=[app_subnet2.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[3]['tag']}])

In [ ]:
data_subnet1 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[4]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[4]['az']
)
ec2_client.create_tags(Resources=[data_subnet1.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[4]['tag']}])

data_subnet2 = ec2_resource.create_subnet(
    CidrBlock=subnet_configs[5]['cidr_block'],
    VpcId=vpc_id,
    AvailabilityZone=subnet_configs[5]['az']
)
ec2_client.create_tags(Resources=[data_subnet2.id],Tags=[{'Key': 'Name', 'Value': subnet_configs[5]['tag']}])

##### Internet Gateway & NAT Gateway

In [ ]:
# Create Internet Gateway and attach that with VPC
rr_igw_id = ec2_resource.create_internet_gateway()
ec2_client.attach_internet_gateway(InternetGatewayId=rr_igw_id, VpcId=vpc_id)

In [ ]:
# Step 1: Allocate an Elastic IP address
rr_eip_allocation_id = ec2_client.allocate_address(Domain="vpc")["AllocationId"]
print(f"Elastic IP allocated: {rr_eip_allocation_id}")

# Step 2: Create the NAT Gateway
rr_nat_gateway = ec2_client.create_nat_gateway(
    SubnetId=public_subnet1.id,
    AllocationId=rr_eip_allocation_id
)
rr_nat_gateway_id = rr_nat_gateway['NatGateway']['NatGatewayId']
print(f"NAT Gateway created: {rr_nat_gateway_id}")

##### Create Route Table and Associte with Subnets

In [ ]:
rr_public_rt = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(Resources=[rr_public_rt.id],Tags=[{'Key': 'Name','Value': 'rr-public-rt'},])

route_params = {'DestinationCidrBlock': '0.0.0.0/0', 'GatewayId': igw_id}
rr_public_rt.create_route(**route_params)

In [ ]:
# Associate Route Table with both public Subnets
rr_public_rt.associate_with_subnet(SubnetId=public_subnet1.id)
rr_public_rt.associate_with_subnet(SubnetId=public_subnet2.id)

In [ ]:
rr_app_subnet1_rt = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(Resources=[rr_app_subnet1_rt.id],Tags=[{'Key': 'Name','Value': 'rr-app-subnet01-rt'},])

rr_app_subnet1_rt.associate_with_subnet(SubnetId=app_subnet1.id)

rr_app_subnet1_rt_params = {'DestinationCidrBlock': '0.0.0.0/0', 'GatewayId': nat_gateway_id}
rr_app_subnet1_rt.create_route(**rr_app_subnet1_rt_params)

In [ ]:
rr_app_subnet2_rt = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(Resources=[rr_app_subnet2_rt.id],Tags=[{'Key': 'Name','Value': 'rr-app-subnet02-rt'},])

rr_app_subnet2_rt.associate_with_subnet(SubnetId=app_subnet2.id)

rr_app_subnet2_rt_params = {'DestinationCidrBlock': '0.0.0.0/0', 'GatewayId': nat_gateway_id}
rr_app_subnet2_rt.create_route(**rr_app_subnet2_rt_params)

In [ ]:
rr_data_subnet1_rt = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(Resources=[rr_data_subnet1_rt.id],Tags=[{'Key': 'Name','Value': 'rr-app-subnet01-rt'},])

rr_data_subnet1_rt.associate_with_subnet(SubnetId=data_subnet1.id)

rr_data_subnet1_rt_params = {'DestinationCidrBlock': '0.0.0.0/0', 'GatewayId': nat_gateway_id}
rr_data_subnet1_rt.create_route(**rr_data_subnet1_rt_params)

In [ ]:
rr_data_subnet2_rt = ec2_resource.create_route_table(VpcId=vpc_id)
ec2_client.create_tags(Resources=[rr_data_subnet2_rt.id],Tags=[{'Key': 'Name','Value': 'rr-app-subnet02-rt'},])

rr_data_subnet2_rt.associate_with_subnet(SubnetId=data_subnet2.id)

rr_data_subnet2_rt_params = {'DestinationCidrBlock': '0.0.0.0/0', 'GatewayId': nat_gateway_id}
rr_data_subnet2_rt.create_route(**rr_data_subnet2_rt_params)

##### Security Group

In [ ]:
rr_alb_sg_rules = [
    {
        "IpProtocol": "tcp",
        "FromPort": 80,  # HTTP
        "ToPort": 80,
        "IpRanges": [{"CidrIp": "0.0.0.0/0", "Description": "HTTP_Port"}],
    }
]

In [ ]:
rr_alb_sg_id = ec2.create_security_group(
    "rr_alb_sg",
    vpc_id,
    inbound_rules=rr_alb_sg_rules,
    outbound_rules="",
    tags=[{"Key": "Name", "Value": "rr_alb_sg"}],
    description="rr_alb_sg",
)["GroupId"]


In [ ]:
rr_app_sg_rules = [
    {
        "IpProtocol": "tcp",
        "FromPort": 80,  # HTTP
        "ToPort": 80,
        "UserIdGroupPairs": [
            {
                "GroupId": rr_alb_sg_id,
                "Description": "Allow tcp traffic only from ALB security group (rr-alb-sg) on port 80",
            }
        ],
    }
]

In [ ]:
rr_app_sg_id = ec2.create_security_group(
    "rr_app_sg",
    vpc_id,
    inbound_rules=rr_app_sg_rules,
    outbound_rules="",
    tags=[{"Key": "Name", "Value": "rr_app_sg"}],
    description="rr_app_sg",
)["GroupId"]


In [ ]:
rr_data_sg_rules = [
    {
        "IpProtocol": "tcp",
        "FromPort": 3306,  # MySQL/Aurora
        "ToPort": 3306,
        "UserIdGroupPairs": [
            {
                "GroupId": rr_app_sg_id,
                "Description": "Allow tcp traffic only from App security group (rr-app-sg) on port 3306",
            }
        ],
    }
]

In [ ]:
rr_data_sg_id = ec2.create_security_group(
    "rr_data_sg",
    vpc_id,
    inbound_rules=rr_data_sg_rules,
    outbound_rules="",
    tags=[{"Key": "Name", "Value": "rr_data_sg"}],
    description="rr_data_sg",
)["GroupId"]

rr_data_sg_rules = [
    {
        "IpProtocol": "tcp",
        "FromPort": 3306,  # MySQL/Aurora
        "ToPort": 3306,
        "UserIdGroupPairs": [
            {
                "GroupId": rr_data_sg_id, # Allow traffic from itself
                "Description": "Allow tcp traffic from itself on port 3306",
            }
        ],
    }
]

# Authorize the rules
ec2_client.authorize_security_group_ingress(
    GroupId=rr_data_sg_id, IpPermissions=rr_data_sg_rules
)

##### Amazon Elastic Container Registry (ECR)

In [ ]:
def create_ecr_repository(repository_name, scan_on_push=False, tag_mutability="MUTABLE"):
    """
    Creates an Amazon Elastic Container Registry (ECR) repository.

    :param repository_name: Name of the ECR repository.
    :param scan_on_push: Whether to enable image scanning on push (default: True).
    :param tag_mutability: Tag mutability setting, 'MUTABLE' or 'IMMUTABLE' (default: 'MUTABLE').
    :return: The response from AWS with details of the created repository.
    """
    ecr_client = boto3.client("ecr")

    try:
        response = ecr_client.create_repository(
            repositoryName=repository_name,
            imageScanningConfiguration={"scanOnPush": scan_on_push},
            imageTagMutability=tag_mutability,
        )
        return response
    except Exception as e:
        print(f"Error creating ECR repository: {e}")
        return None



In [ ]:
# Example Usage
repo_name = "ritual-roast"
response = create_ecr_repository(repo_name)
if response:
    print("Repository Created:", response)

##### IAM Role

In [ ]:
EC2_ROLE_NAME = "iam-role-grant-ec2-ssm-and-ecr-access"

In [ ]:
assume_role_policy_document = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["sts:AssumeRole"],
            "Principal": {"Service": ["ec2.amazonaws.com"]},
        }
    ],
}


In [ ]:
# Create the IAM role with the assume role policy document
EC2_ROLE_ARN = iam_client.create_role(
    RoleName=EC2_ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(assume_role_policy_document),
)["Role"]["Arn"]


In [ ]:
policy_arns = [
    # AWS Systems Manager (SSM) allow us to communicate with the instance over Session Manager.
    "arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore",
    # Amazon ECR policy to allow the instance to push/pull images from the ECR repository.
    "arn:aws:iam::aws:policy/EC2InstanceProfileForImageBuilderECRContainerBuilds",
]
[iam_client.attach_role_policy(RoleName=EC2_ROLE_NAME, PolicyArn=arn) for arn in policy_arns]

##### EC2 Instance

In [ ]:
# Launch EC2 instance with tagging using TagSpecifications
response = ec2_client.run_instances(
    ImageId=AWS_DEFAULT_IMAGE_ID,  # Amazon Linux Machine Image (AMI)
    InstanceType="t2.micro",  #'t2.medium', 't2.micro'
    MinCount=1,
    MaxCount=1,
    KeyName="AMShah",  # Replace with your key pair
    TagSpecifications=[
        {
            "ResourceType": "instance",
            "Tags": [{"Key": "Name", "Value": "rr-docker-build-server"}],
        }
    ],
    BlockDeviceMappings=[
        {
            "DeviceName": "/dev/sda1",  # Default root volume
            "Ebs": {
                "VolumeSize": 20,  # Volume size in GiB
                "VolumeType": "gp2",  # General Purpose SSD
            },
        }
    ],
    SecurityGroupIds=[ALL_IN_ONE_SG],
    SubnetId=app_subnet1.id,
    IamInstanceProfile={"Arn": EC2_ROLE_ARN, "Name": EC2_ROLE_NAME},
    # IamInstanceProfile={"Name": "EC2InstanceProfileForImageBuilderECRContainerBuilds"},
)

# Extract the instance ID of the newly created instance
instance_id = response["Instances"][0]["InstanceId"]
print(f"Instance created with ID: {instance_id}")


- **Build image in EC2 Instance and push it into ECR**
  -   Connect the instance over Session Manager

-   `$ sudo su - ec2-user` -> Chnage the user contex to local user account (`ec2-user`)
-   `$ sudo yum install docker` -> Install Docker.
-   `$ sudo service docker start` -> Start the Docker servic on the instance.
-   `$ sudo usermod -a -G docker ec2-user` -> Add the current user (`ec2-user`) into the `docker` group.
-   `$ exit` -> Exit and Reconnect to the instance to have the effect of it's modification.
-   `$ sudo su - ec2-user` -> Chnage the user contex to local user account (`ec2-user`) again after reconnect.

-   `$ wget https://github.com/iaasacademy/aws-how-to-guide/raw/238deeefb955ddef46c673f5154754f679410d57/amazon-ecs-mini-project/ritual-roast-code.zip`
    -   Download the ziped file of source code from Git repository
-   `$ unzip ritual-roast-code.zip`
-   `$ docker build -t your_ECR_URI .`
-   `$ docker images`

-   **Push Image to ECR usin AWS CLI**

##### RDS

In [ ]:
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_USERNAME = os.environ["USERNAME"]
DB_PASSWORD = os.environ["PASSWORD"]

In [ ]:
rds.create_subnet_group(subnet_group_name="rr-db-subnet-group", subnet_ids=[data_subnet1.id, data_subnet2.id])


In [ ]:
db_param = {
    "db_instance_identifier": "ritualroastdb",
    "db_name": "ritualroastdb",
    "db_username": DB_USERNAME,
    "db_password": DB_PASSWORD,
    "engine": "mysql",
    "port": 3306,
    "engine_version": "8.0.32",
    "db_instance_class": "db.t3.micro",
    "allocated_storage": 20,
    "availability_zone": "us-east-1a",
    "tags": [{"Key": "Project", "Value": "ritualroastdb"}],
    "security_group_ids": [rr_data_sg_id],
    "db_subnet_group_name": "rr-db-subnet-group",
}

In [ ]:
rds.create_rds_instance(**db_param)

##### SMS (Secret Management Service)

In [ ]:
# Initialize the Secrets Manager client
sms_client = boto3.client('secretsmanager', region_name='us-east-1')  # Replace 'us-east-1' with your AWS region


# Define secret name and value
secret_name = "prod/db_password"
secret_value = '{"username": "admin", "password": "mysecretpassword"}'

In [ ]:
response = sms_client.create_secret(
    Name=secret_name,
    Description="This is a sample secret for database credentials",
    SecretString=secret_value
)

In [ ]:
# Retrieve the secret value
response = sms_client.get_secret_value(SecretId=secret_name)

# The secret can be either a JSON string or a plain text value
if 'SecretString' in response:
    secret = response['SecretString']
    # If it's JSON formatted, load it into a Python dictionary
    secret =  json.loads(secret)
elif 'SecretBinary' in response:
    # Decode binary secrets
    secret = response['SecretBinary']


In [ ]:
response = sms_client.delete_secret(
    SecretId='string',
    # RecoveryWindowInDays=123,
    ForceDeleteWithoutRecovery=True # You can’t use both this parameter and RecoveryWindowInDays
)